# Create submission files

For a simulations with ekbatch you need an init file containing the stimuli and the CVs of the regions from a CARP simulation, you will need:
-  vtx file for a stimulus
-  a set of tags with the conduction velocities

In [1]:
import json
import numpy as np

def json_to_init(stimuli, tag_file, json_param_file, init_file_name):

    # Read tags
    f_input = open(tag_file,"r")
    tags = json.load(f_input)
    f_input.close()

    # Read CVs
    f_input = open(json_param_file,"r")
    params = json.load(f_input)
    f_input.close()

    tags_ventricles_names = ["LV", "RV"]
    CV_ventricle_name = "CV_ventricles"
    k_ventricles_name = "k_ventricles"

    tags_FEC_names = ["FEC_LV", "FEC_RV", "FEC_SV"]
    k_FEC_name = "k_FEC"

    tags_atria_names = ["LA", "RA"]
    CV_atria_name = "CV_atria"
    k_atria_name = "k_atria"

    tags_bachmann_names = ["BB"]
    k_BB_name = "k_BB"

    vtx = []
    nVtx = 0

    for vtxFile in stimuli:
        temp = np.loadtxt(vtxFile, dtype=int, skiprows=2, ndmin=1)
        vtx.append(temp)
        nVtx += temp.shape[0]

    # write .init file
    f = open(init_file_name,'w')

    # header
    f.write('vf:0 vs:0 vn:0 vPS:0\n') # Default properties for tags not specified
    f.write('retro_delay:0 antero_delay:0\n') # If there's no 1D purkinje system, it's ignored.
    # number of stimuli and regions
    f.write('%d %d\n' % (int(nVtx), int(len(tags_ventricles_names)) + len(tags_FEC_names) + len(tags_atria_names) + len(tags_bachmann_names)))
    # stimulus
    for i in range(len(vtx)):
        if len(vtx[i]) == 1:
            f.write('%d %f\n' % (vtx[i],0))
        else:
            for n in vtx[i]:
                f.write('%d %f\n' % (int(n),0))
                
    return_tags_str = ''
    # ek regions
    for i,tag_name in enumerate(tags_ventricles_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_ventricle_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_ventricles_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_ventricles_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    for i,tag_name in enumerate(tags_FEC_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_FEC_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_FEC_name]), 
                                   float(params["EP"][CV_ventricle_name])*float(params["EP"][k_FEC_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    for i,tag_name in enumerate(tags_atria_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_atria_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_atria_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_atria_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    for i,tag_name in enumerate(tags_bachmann_names):
        f.write('%d %f %f %f\n' % (int(tags[tag_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_BB_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_BB_name]), 
                                   float(params["EP"][CV_atria_name])*float(params["EP"][k_BB_name])))
        return_tags_str += ',' + str(int(tags[tag_name]))

    f.close()
    
    return return_tags_str[1:]

In [3]:
import os

heart_folder = "/media/croderog/SeagateExpansionDrive/HCM/10KH00011/"
scenario = 13
Nsim = 180

stimuli = [f'{heart_folder}/sims_folder/fascicles_lv.vtx',
                f'{heart_folder}/sims_folder/fascicles_rv.vtx',
                f'{heart_folder}/sims_folder/SAN.vtx']

json_param_path        = f'{heart_folder}/scenarios/{scenario}/json_files/'
tag_file        = f'{json_param_path}/tags.json'
init_file_path  = f'{heart_folder}/scenarios/{scenario}/data/init_files'

os.system("mkdir -p " + init_file_path)

for sim_num in range(Nsim):
    tags_activated = json_to_init(stimuli=stimuli,
                tag_file=tag_file,
                json_param_file=os.path.join(json_param_path,str(sim_num) + '.json'),
                init_file_name=os.path.join(init_file_path,str(sim_num) + '.init')
                )

KeyError: 'LV'

# Run simulations

In [9]:
for heart_num in range(1,20):

    if heart_num != 15:

        if heart_num < 10:
            heart = f"0{heart_num}"
        else:
            heart = f"{heart_num}"

        sims_folder = '/media/croderog/SeagateExpansionDrive/rodero_healthy/h' + heart + '/EP_same_X/output/simulations'

        meshname = '/media/croderog/SeagateExpansionDrive/rodero_healthy/h' + heart + '/pre_simulation/myocardium_AV_FEC_BB_lvrv'
        init_file_path = '/media/croderog/SeagateExpansionDrive/rodero_healthy/h' + heart + '/EP_same_X/data/init_files'


        cmd = ['ekbatch',meshname]
        init_cmd = ','.join([os.path.join(init_file_path,str(sim_num)) for sim_num in range(180)])

        os.system(' '.join(cmd+[init_cmd] + [tags_activated]))

        os.makedirs(sims_folder,exist_ok=True)
        for sim_num in range(180):
            os.system('mv ' + os.path.join(init_file_path,str(sim_num) + '.dat ') + sims_folder)


Executable ID: KCL_LHR_CARPENTRY
Found license file path: /home/common/CARPentry_KCL_latest/license/license.bin
Using OpenMP parallelization with 24 threads.
Reading mesh ..
Reading elements (txt):                           [==============================]
Reading points (txt):                             [==============================]
Reading fibers (txt):                             [==============================]
Needed 12.4871 seconds for mesh-reading and subdomain-extraction

The simulation domain consists of:
2218842	elements
447591	nodes

Parsed init file: /media/croderog/SeagateExpansionDrive/rodero_healthy/h01/EP_same_X/data/init_files/0.init
The used velocities (in m/s) are:
Fiber direction:	0
Sheet direction:	0
Normal direction:	0
Purkinje system:	0
The used junction delays (in ms) are:
Anterograde delay:	0
Retrograde delay:	0

Solving ..
Eikonal solve progress:                           [==============================]
Needed 2.42831 seconds
Wrote /media/croderog/Seagate

# Extract the output

In [1]:
# Extracted from Marina's library

def electrophysiology_output(basefolder,
							 elem_file,
							 tags,
	   						 start_sample=0,
	   						 last_sample=1,
	   						 output_file='Y.txt'):

	print('Reading mesh elem file...')
	elem = np.loadtxt(elem_file,dtype=int,usecols=[1,2,3,4,5],skiprows=1)
	print('Done.')

	V_EIDX = np.where(np.isin(elem[:,-1],tags["ventricles"]+tags["fast_endo"])==1)[0]
	A_EIDX = np.where(np.isin(elem[:,-1],tags["atria"]+tags["bachmann_bundle"])==1)[0]

	V_VTX = np.unique(elem[V_EIDX,0:4].flatten())
	A_VTX = np.unique(elem[A_EIDX,0:4].flatten())

	output = np.zeros((last_sample-start_sample+1,2))

	count = 0
	for i in range(start_sample,last_sample+1):
		print('Computing output for '+str(i)+'.dat...')
		AT=np.loadtxt(os.path.join(basefolder,str(i)+".dat"),dtype=float)
		if (np.min(AT[V_VTX]<0)):
			raise Exception("The ventricles contain a negative activation time.")
		if (np.min(AT[A_VTX]<0)):
			raise Exception("The atria contain a negative activation time.")
			
		output[count,0] = np.max(AT[A_VTX])-np.min(AT[A_VTX])
        
		output[count,1] = np.max(AT[V_VTX])-np.min(AT[V_VTX])
		count += 1

	np.savetxt(output_file,output,fmt="%g")

In [7]:
import json
import numpy as np
import os

tag_file        = '/home/croderog/Desktop/IC_projects/barrows_preprocessing/parfiles/tags_lvrv.json'

for heart_num in range(1,20):

	if heart_num != 15:

		if heart_num < 10:
			heart = f"0{heart_num}"
		else:
			heart = f"{heart_num}"

		basefolder = '/media/croderog/SeagateExpansionDrive/rodero_healthy/h' + heart + '/EP_same_X/output/simulations'
		elem_file = '/media/croderog/SeagateExpansionDrive/rodero_healthy/h' + heart + '/pre_simulation/myocardium_AV_FEC_BB_lvrv.elem'

		f_input = open(tag_file,"r")
		tags = json.load(f_input)
		f_input.close()


		tags_modified = tags.copy()
		tags_modified["ventricles"] = [tags_modified["LV"], tags_modified["RV"]]
		tags_modified["fast_endo"] = [tags_modified["FEC_RV"], tags_modified["FEC_SV"]]
		tags_modified["atria"] = [tags_modified["LA"], tags_modified["RA"]]
		tags_modified["bachmann_bundle"] = [tags_modified["BB"]]

		output_path = '/media/croderog/SeagateExpansionDrive/rodero_healthy/h' + heart + '/EP_same_X/output'

		os.makedirs(output_path,exist_ok=True)

		electrophysiology_output(basefolder=basefolder,
									elem_file=elem_file,
									tags=tags_modified,
									start_sample=0,
									last_sample=179,
									output_file=os.path.join(output_path,'Y.txt'))

Reading mesh elem file...
Done.
Computing output for 0.dat...
Computing output for 1.dat...
Computing output for 2.dat...
Computing output for 3.dat...
Computing output for 4.dat...
Computing output for 5.dat...
Computing output for 6.dat...
Computing output for 7.dat...
Computing output for 8.dat...
Computing output for 9.dat...
Computing output for 10.dat...
Computing output for 11.dat...
Computing output for 12.dat...
Computing output for 13.dat...
Computing output for 14.dat...
Computing output for 15.dat...
Computing output for 16.dat...
Computing output for 17.dat...
Computing output for 18.dat...
Computing output for 19.dat...
Computing output for 20.dat...
Computing output for 21.dat...
Computing output for 22.dat...
Computing output for 23.dat...
Computing output for 24.dat...
Computing output for 25.dat...
Computing output for 26.dat...
Computing output for 27.dat...
Computing output for 28.dat...
Computing output for 29.dat...
Computing output for 30.dat...
Computing output 